### Import Libraries

In [3]:
! pip install transformers
! pip install datasets

In [25]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,AutoModelForMaskedLM
from datasets import Dataset
import re
import torch
import warnings
warnings.filterwarnings('ignore')

### Read & Explore Data

In [16]:
df = pd.read_csv('Final_Data.csv')
df.head()

,review_description,rating,company
0,رائع,positive,talbat
1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشك...,positive,talbat
2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال با...,negative,talbat
3,لماذا لا يمكننا طلب من ماكدونالدز؟,negative,talbat
4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكو...,negative,talbat


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40046 entries, 0 to 40045
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   review_description  40045 non-null  object
 1   rating              40046 non-null  object
 2   company             40046 non-null  object
dtypes: object(3)
memory usage: 938.7+ KB


In [18]:
df.drop("company", axis=1, inplace=True)

In [19]:
df.isna().sum()

,0
review_description,1
rating,0


In [20]:
df.dropna(inplace=True)

In [21]:
df.duplicated().sum()

np.int64(941)

In [22]:
df.drop_duplicates(inplace=True)

In [23]:
df['rating'] = df['rating'].map({'positive': 1, 'negative': 0})

In [24]:
df['rating'].value_counts()

,count
rating,
1.0,23211
0.0,13975


### Text Preprocessing 

In [27]:
df.rename(columns={'review_description': 'text', 'rating': 'labels'}, inplace=True)

In [26]:
def preprocess_for_transformer(text):
    text = re.sub(r'<.*?>', '', text)          
    text = re.sub(r'http\S+|www\S+', '', text) 
    text = re.sub(r'@\w+|#\w+', '', text)      
    text = re.sub(r'(.)\1{2,}', r'\1\1', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [28]:
df['text'] = df['text'].apply(preprocess_for_transformer)